# Model_Training_01 — RepCount Baseline (Rebuilt)

This notebook trains a baseline temporal rep counter from extracted pose features.


In [ ]:
# Optional (uncomment if needed in a fresh environment)
# !pip install torch numpy pandas ultralytics


## 1) Objective and Evaluation

**Objective**
- Train a baseline temporal regressor on cleaned train/valid annotations.
- Use pre-extracted pose features (`pose_feature_index.csv` + `.npy/.npz`).

**Primary metrics**
- MAE, RMSE, Within-1 accuracy, and per-class MAE.


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Libraries loaded. Seed set to', SEED)


## 2) Resolve Paths and Load Prepared Data

This section resolves paths robustly (independent of notebook working directory).


In [ ]:
def resolve_project_dir() -> Path:
    # Resolve .../CV_Image_pose_detection robustly from current working directory.
    cwd = Path.cwd().resolve()

    # Case A: running from repo root or any parent where CV_Image_pose_detection is a child
    for base in [cwd, *cwd.parents]:
        cand = (base / 'CV_Image_pose_detection').resolve()
        if (cand / 'Data' / 'LLSP').exists() and (cand / 'artifacts').exists():
            return cand

    # Case B: running from inside CV_Image_pose_detection subtree
    for base in [cwd, *cwd.parents]:
        if (base / 'Data' / 'LLSP').exists() and (base / 'artifacts').exists():
            return base.resolve()

    raise FileNotFoundError('Could not resolve project directory containing Data/LLSP and artifacts.')


PROJECT_DIR = resolve_project_dir()
NOTEBOOK_DIR = PROJECT_DIR / 'artifacts' / '3_Modeling'
DATA_DIR = PROJECT_DIR / 'Data' / 'LLSP' / 'annotation_cleaned'

TRAIN_CLEAN = DATA_DIR / 'train_cleaned.csv'
VALID_CLEAN = DATA_DIR / 'valid_cleaned.csv'
FEATURE_INDEX = DATA_DIR / 'pose_feature_index.csv'
CLASS_W = DATA_DIR / 'class_weights_train.csv'
SAMPLE_W = DATA_DIR / 'train_sample_weights.csv'
MANIFEST = DATA_DIR / 'decisions_manifest.json'

required_files = [TRAIN_CLEAN, VALID_CLEAN, FEATURE_INDEX]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError('Missing required input files:\\n' + '\\n'.join(missing_files))

train_df = pd.read_csv(TRAIN_CLEAN)
valid_df = pd.read_csv(VALID_CLEAN)
feat_idx_df = pd.read_csv(FEATURE_INDEX)
class_w_df = pd.read_csv(CLASS_W) if CLASS_W.exists() else None
sample_w_df = pd.read_csv(SAMPLE_W) if SAMPLE_W.exists() else None
manifest = json.load(open(MANIFEST)) if MANIFEST.exists() else {}

# Keep only columns needed for training metadata.
train_meta = train_df[['name', 'type', 'count']].copy()
valid_meta = valid_df[['name', 'type', 'count']].copy()

print('Resolved PROJECT_DIR:', PROJECT_DIR)
print('Resolved DATA_DIR   :', DATA_DIR)
print(f'train rows={len(train_meta)}, valid rows={len(valid_meta)}')
print('feature_index rows  =', len(feat_idx_df))
print('manifest loaded     =', bool(manifest))


In [ ]:
required_cols = {'name', 'type', 'count'}
missing_train = required_cols - set(train_meta.columns)
missing_valid = required_cols - set(valid_meta.columns)
if missing_train or missing_valid:
    raise ValueError(f'Missing required columns - train:{missing_train}, valid:{missing_valid}')

train_names = set(train_meta['name'].astype(str).str.strip().str.lower())
valid_names = set(valid_meta['name'].astype(str).str.strip().str.lower())
name_overlap = sorted(train_names & valid_names)

print('Required columns present.')
print('Exact train-valid name overlap:', len(name_overlap))
if name_overlap:
    print('Sample overlaps:', name_overlap[:10])


## 3) Modeling Design

- Input: temporal pose features per video `[T, F]`.
- Model: mean-pool over time + MLP regressor.
- Target: rep count regression.


## 4) Training Config


In [ ]:
CFG = {
    'run_name': 'baseline_v2_rebuilt',
    'epochs': 30,
    'batch_size': 16,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_workers': 0,
    'device': 'cuda',

    # feature tensor shape target
    'seq_len': 64,
    'feat_dim': 64,
    'hidden_dim': 128,
    'dropout': 0.2,

    # I/O
    'feature_index_path': str(FEATURE_INDEX),
    'output_root': str((NOTEBOOK_DIR / 'training_outputs').resolve()),

    # quality gates
    'min_train_rows': 32,
    'min_valid_rows': 16,

    # optional weighting
    'use_class_weights': False,
    'use_weighted_sampler': False,
}

CFG


In [ ]:
class_names = sorted(train_meta['type'].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

train_meta = train_meta.assign(class_idx=train_meta['type'].map(class_to_idx).astype(int))
valid_meta = valid_meta.assign(class_idx=valid_meta['type'].map(class_to_idx).astype(int))

print('num_classes =', len(class_names))
print(class_to_idx)


## 5) PyTorch Setup and Model Skeleton


In [ ]:
TORCH_AVAILABLE = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
except Exception as e:
    TORCH_AVAILABLE = False
    print('PyTorch not available:', e)

if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device('cuda' if (CFG['device'] == 'cuda' and torch.cuda.is_available()) else 'cpu')
    print('Using device:', device)


In [ ]:
if TORCH_AVAILABLE:
    class TemporalCountRegressor(nn.Module):
        def __init__(self, feat_dim=64, hidden_dim=128, dropout=0.2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1),
            )

        def forward(self, x):
            x = x.mean(dim=1)
            y = self.net(x)
            return y.squeeze(-1)

    model = TemporalCountRegressor(
        feat_dim=CFG['feat_dim'],
        hidden_dim=CFG['hidden_dim'],
        dropout=CFG['dropout'],
    ).to(device)

    print(model.__class__.__name__)


## 6) Metrics


In [ ]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def within_1_acc(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred) <= 1.0))


def per_class_mae(df_eval, y_pred_col='pred_count'):
    return (
        df_eval
        .assign(abs_err=lambda d: np.abs(d['count'].astype(float) - d[y_pred_col].astype(float)))
        .groupby('type', as_index=False)['abs_err']
        .mean()
        .rename(columns={'abs_err': 'mae'})
        .sort_values('mae', ascending=False)
    )


## 7) Training Pipeline

Includes strict prechecks for feature-index alignment and feature-file existence.


In [ ]:
from datetime import datetime, timezone

TRAIN_RESULTS = None

if not TORCH_AVAILABLE:
    print('Skip training pipeline: PyTorch is not available.')
else:
    feature_index_path = Path(CFG['feature_index_path']).resolve()
    if not feature_index_path.exists():
        raise FileNotFoundError(f'Feature index not found: {feature_index_path}')

    output_dir = Path(CFG['output_root']) / CFG['run_name']
    ckpt_dir = output_dir / 'checkpoints'
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    feat_idx = pd.read_csv(feature_index_path)
    required_feat_cols = {'name', 'feature_path'}
    missing_feat_cols = required_feat_cols - set(feat_idx.columns)
    if missing_feat_cols:
        raise ValueError(f'pose_feature_index.csv missing columns: {missing_feat_cols}')

    feat_idx = feat_idx.copy()
    feat_idx['name_norm'] = feat_idx['name'].astype(str).str.strip().str.lower()

    train_local = train_meta.copy()
    valid_local = valid_meta.copy()
    train_local['name_norm'] = train_local['name'].astype(str).str.strip().str.lower()
    valid_local['name_norm'] = valid_local['name'].astype(str).str.strip().str.lower()

    train_m = train_local.merge(feat_idx[['name_norm', 'feature_path']], on='name_norm', how='left')
    valid_m = valid_local.merge(feat_idx[['name_norm', 'feature_path']], on='name_norm', how='left')

    train_missing_index = int(train_m['feature_path'].isna().sum())
    valid_missing_index = int(valid_m['feature_path'].isna().sum())

    train_m = train_m.dropna(subset=['feature_path']).reset_index(drop=True)
    valid_m = valid_m.dropna(subset=['feature_path']).reset_index(drop=True)

    train_exists = train_m['feature_path'].map(lambda p: Path(p).exists())
    valid_exists = valid_m['feature_path'].map(lambda p: Path(p).exists())

    train_missing_files = int((~train_exists).sum())
    valid_missing_files = int((~valid_exists).sum())

    train_m = train_m.loc[train_exists].reset_index(drop=True)
    valid_m = valid_m.loc[valid_exists].reset_index(drop=True)

    alignment_report = {
        'train_total_rows': int(len(train_local)),
        'valid_total_rows': int(len(valid_local)),
        'train_missing_index_entries': train_missing_index,
        'valid_missing_index_entries': valid_missing_index,
        'train_missing_feature_files': train_missing_files,
        'valid_missing_feature_files': valid_missing_files,
        'train_rows_after_alignment': int(len(train_m)),
        'valid_rows_after_alignment': int(len(valid_m)),
    }
    with open(output_dir / 'feature_alignment_report.json', 'w', encoding='utf-8') as f:
        json.dump(alignment_report, f, ensure_ascii=True, indent=2)

    print('Feature alignment summary:')
    for k, v in alignment_report.items():
        print(f'  {k}: {v}')

    if len(train_m) < CFG['min_train_rows'] or len(valid_m) < CFG['min_valid_rows']:
        raise RuntimeError(
            f"Insufficient aligned rows for training. "
            f"train={len(train_m)} (min {CFG['min_train_rows']}), "
            f"valid={len(valid_m)} (min {CFG['min_valid_rows']}). "
            "Re-run pose feature extraction for the latest cleaned splits."
        )

    def _resolve_feature_path(p):
        pp = Path(p)
        if not pp.is_absolute():
            pp = (DATA_DIR / pp).resolve()
        return pp

    def _load_feature_array(path):
        path = _resolve_feature_path(path)
        if not path.exists():
            raise FileNotFoundError(f'Feature file not found: {path}')

        if path.suffix.lower() == '.npy':
            arr = np.load(path)
        elif path.suffix.lower() == '.npz':
            z = np.load(path)
            if 'x' in z:
                arr = z['x']
            elif 'features' in z:
                arr = z['features']
            else:
                arr = z[z.files[0]]
        else:
            raise ValueError(f'Unsupported feature extension: {path.suffix}')

        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr[:, None]
        if arr.ndim != 2:
            raise ValueError(f'Expected [T,F], got shape={arr.shape} for {path}')
        return arr

    def _fix_shape(arr, seq_len, feat_dim):
        if arr.shape[0] >= seq_len:
            arr = arr[:seq_len, :]
        else:
            pad_t = np.zeros((seq_len - arr.shape[0], arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad_t], axis=0)

        if arr.shape[1] >= feat_dim:
            arr = arr[:, :feat_dim]
        else:
            pad_f = np.zeros((arr.shape[0], feat_dim - arr.shape[1]), dtype=np.float32)
            arr = np.concatenate([arr, pad_f], axis=1)
        return arr

    class RepCountFeatureDataset(Dataset):
        def __init__(self, df, seq_len, feat_dim):
            self.df = df.reset_index(drop=True)
            self.seq_len = seq_len
            self.feat_dim = feat_dim

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            arr = _load_feature_array(row['feature_path'])
            arr = _fix_shape(arr, self.seq_len, self.feat_dim)

            x = torch.tensor(arr, dtype=torch.float32)
            y = torch.tensor(float(row['count']), dtype=torch.float32)
            c = torch.tensor(int(row['class_idx']), dtype=torch.long)
            meta = {'name': str(row['name']), 'type': str(row['type'])}
            return x, y, c, meta

    train_ds = RepCountFeatureDataset(train_m, CFG['seq_len'], CFG['feat_dim'])
    valid_ds = RepCountFeatureDataset(valid_m, CFG['seq_len'], CFG['feat_dim'])

    sampler = None
    if CFG['use_weighted_sampler'] and sample_w_df is not None:
        weight_col_candidates = [c for c in sample_w_df.columns if 'weight' in c.lower()]
        if 'name' in sample_w_df.columns and weight_col_candidates:
            w_col = weight_col_candidates[0]
            w_map = (
                sample_w_df[['name', w_col]]
                .assign(name=lambda d: d['name'].astype(str).str.strip().str.lower())
                .drop_duplicates('name')
                .set_index('name')[w_col]
                .to_dict()
            )
            train_weights = train_m['name_norm'].map(lambda x: float(w_map.get(x, 1.0))).values
            sampler = WeightedRandomSampler(
                weights=torch.tensor(train_weights, dtype=torch.double),
                num_samples=len(train_weights),
                replacement=True,
            )
            print(f'Weighted sampler enabled using: {w_col}')
        else:
            print('Weighted sampler requested but sample-weight schema is incompatible; fallback to shuffle.')

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG['batch_size'],
        shuffle=(sampler is None),
        sampler=sampler,
        num_workers=CFG['num_workers'],
        drop_last=False,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG['batch_size'],
        shuffle=False,
        num_workers=CFG['num_workers'],
        drop_last=False,
    )

    class_weight_tensor = None
    if CFG['use_class_weights'] and class_w_df is not None and {'type', 'inv_freq_weight'}.issubset(class_w_df.columns):
        cw_map = class_w_df.set_index('type')['inv_freq_weight'].to_dict()
        class_weight_tensor = torch.tensor(
            [float(cw_map.get(idx_to_class[i], 1.0)) for i in range(len(idx_to_class))],
            dtype=torch.float32,
            device=device,
        )
        print('Class-weighted MAE enabled.')

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

    def _weighted_mae_loss(pred, target, class_idx):
        err = torch.abs(pred - target)
        if class_weight_tensor is not None:
            err = err * class_weight_tensor[class_idx]
        return err.mean()

    best_valid_mae = float('inf')
    best_epoch = -1
    history = []
    best_pred_df = None
    best_pc_df = None

    for epoch in range(1, CFG['epochs'] + 1):
        model.train()
        train_losses = []

        for xb, yb, cb, _meta in train_loader:
            xb, yb, cb = xb.to(device), yb.to(device), cb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = _weighted_mae_loss(pred, yb, cb)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu().item()))

        train_loss = float(np.mean(train_losses)) if train_losses else np.nan

        model.eval()
        all_pred, all_true, all_type, all_name = [], [], [], []
        valid_losses = []
        with torch.no_grad():
            for xb, yb, cb, meta in valid_loader:
                xb, yb, cb = xb.to(device), yb.to(device), cb.to(device)
                pred = model(xb)
                loss = _weighted_mae_loss(pred, yb, cb)
                valid_losses.append(float(loss.detach().cpu().item()))

                all_pred.extend(pred.detach().cpu().numpy().tolist())
                all_true.extend(yb.detach().cpu().numpy().tolist())
                all_type.extend(meta['type'])
                all_name.extend(meta['name'])

        valid_loss = float(np.mean(valid_losses)) if valid_losses else np.nan
        valid_mae = mae(all_true, all_pred)
        valid_rmse = rmse(all_true, all_pred)
        valid_w1 = within_1_acc(all_true, all_pred)

        eval_df = pd.DataFrame({
            'name': all_name,
            'type': all_type,
            'count': all_true,
            'pred_count': all_pred,
        })
        eval_df['abs_err'] = (eval_df['count'] - eval_df['pred_count']).abs()
        pc_df = per_class_mae(eval_df, y_pred_col='pred_count')

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'valid_loss': valid_loss,
            'valid_mae': valid_mae,
            'valid_rmse': valid_rmse,
            'valid_within1': valid_w1,
        })

        if valid_mae < best_valid_mae:
            best_valid_mae = valid_mae
            best_epoch = epoch
            best_pred_df = eval_df.copy()
            best_pc_df = pc_df.copy()
            torch.save(
                {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'cfg': CFG,
                    'class_to_idx': class_to_idx,
                    'best_valid_mae': best_valid_mae,
                },
                ckpt_dir / 'best_model.pt',
            )

        print(
            f"Epoch {epoch:02d}/{CFG['epochs']} | train_loss={train_loss:.4f} | "
            f"valid_loss={valid_loss:.4f} | valid_MAE={valid_mae:.4f} | "
            f"best_MAE={best_valid_mae:.4f} ({best_epoch})"
        )

    history_df = pd.DataFrame(history)
    history_path = output_dir / 'metrics_history.csv'
    pred_path = output_dir / 'valid_predictions_best.csv'
    pc_path = output_dir / 'per_class_mae_best.csv'
    summary_path = output_dir / 'run_summary.json'

    history_df.to_csv(history_path, index=False)
    if best_pred_df is not None:
        best_pred_df.to_csv(pred_path, index=False)
    if best_pc_df is not None:
        best_pc_df.to_csv(pc_path, index=False)

    summary = {
        'run_name': CFG['run_name'],
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'best_epoch': int(best_epoch),
        'best_valid_mae': float(best_valid_mae),
        'epochs_ran': int(CFG['epochs']),
        'num_train_rows_used': int(len(train_m)),
        'num_valid_rows_used': int(len(valid_m)),
        'config': CFG,
        'artifacts': {
            'history_csv': str(history_path),
            'best_model': str(ckpt_dir / 'best_model.pt'),
            'valid_predictions_best_csv': str(pred_path),
            'per_class_mae_best_csv': str(pc_path),
            'feature_alignment_report_json': str(output_dir / 'feature_alignment_report.json'),
        },
    }
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)

    TRAIN_RESULTS = {
        'history_df': history_df,
        'best_pred_df': best_pred_df,
        'best_pc_df': best_pc_df,
        'summary': summary,
    }

    print('\nSaved artifacts:')
    print(' ', history_path)
    print(' ', ckpt_dir / 'best_model.pt')
    print(' ', pred_path)
    print(' ', pc_path)
    print(' ', summary_path)


In [ ]:
if TRAIN_RESULTS is not None:
    print('Best epoch:', TRAIN_RESULTS['summary']['best_epoch'])
    print('Best valid MAE:', TRAIN_RESULTS['summary']['best_valid_mae'])
    print('\nPer-class MAE (best):')
    display(TRAIN_RESULTS['best_pc_df'])
else:
    print('No training results in memory yet.')


## 8) Experiment Log

| run_name | model | feature_source | train_rows_used | valid_rows_used | best_valid_mae | notes |
|---|---|---|---:|---:|---:|---|
| baseline_v2_rebuilt | TemporalCountRegressor | pose_feature_index.csv |  |  |  |  |
